In [ ]:
import torch
import flh
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn

from transformers import LlamaForCausalLM, AutoTokenizer
from flh.quantized_model.modeling_llama import FLH_FP16LlamaForCausalLM, FLH_LlamaForCausalLM

In [ ]:
model_path = "/home/mashaobo/.cache/modelscope/hub/models/LLM-Research/Llama-3.2-1B"

tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
model = LlamaForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="cuda:2",
)

In [ ]:
flh_fp16_model = FLH_FP16LlamaForCausalLM.from_float(model, target_device="cuda:2", fuse_layernorm=True)

In [ ]:
flh_model_gptq = FLH_LlamaForCausalLM.from_float(
    model,
    target_device="cuda:2",
    weight_bits=16,
    act_bits=16,
    weight_group_size=128,
    act_group_size=128,
    weight_sym=True,
    act_sym=False,
    use_gptq=True
)

In [ ]:
flh_model = FLH_LlamaForCausalLM.from_float(
    model,
    target_device="cuda:2",
    weight_bits=16,
    act_bits=16,
    weight_group_size=128,
    act_group_size=128,
    weight_sym=True,
    act_sym=False
)

In [ ]:
print(flh_model_gptq.model.layers[0].self_attn.o_proj.weight)

In [ ]:
o_proj_weight = model.model.layers[0].self_attn.o_proj.weight
o_proj_fp16_weight = flh_fp16_model.model.layers[0].self_attn.o_proj.weight

print(o_proj_weight[0])
print(o_proj_fp16_weight[0])

o_proj_quant = flh_model.model.layers[0].self_attn.o_proj.weight
o_proj_quant = flh.nn.fast_hadamard_transform(o_proj_quant, group_size=128, normalize=True)
o_proj_quant = flh.nn.fast_hadamard_transform(o_proj_quant.t(), group_size=128, normalize=False).t()
print(o_proj_quant[0])


In [ ]:
post_attention_layernorm = model.model.layers[0].post_attention_layernorm.weight
post_attention_layernorm_fp16 = flh_fp16_model.model.layers[0].post_attention_layernorm.weight

print(post_attention_layernorm)
print(post_attention_layernorm_fp16)

up_proj_weight = model.model.layers[0].mlp.up_proj.weight
up_proj_weight_fp16 = flh_fp16_model.model.layers[0].mlp.up_proj.weight

print(up_proj_weight * post_attention_layernorm)
print(up_proj_weight_fp16)

up_proj_weight_quant = flh_model.model.layers[0].mlp.up_proj.weight

print(flh.nn.fast_hadamard_transform(up_proj_weight_quant, group_size=128, normalize=True))

In [ ]:
down_proj_weight = model.model.layers[0].mlp.down_proj.weight
down_proj_weight_fp16 = flh_fp16_model.model.layers[0].mlp.down_proj.weight

print(down_proj_weight)
print(down_proj_weight_fp16)

down_proj_weight_quant = flh_model.model.layers[0].mlp.down_proj.weight
down_proj_weight_quant = flh.nn.fast_hadamard_transform(down_proj_weight_quant, group_size=128, normalize=True)
down_proj_weight_quant = flh.nn.fast_hadamard_transform(down_proj_weight_quant.t(), group_size=128, normalize=False).t()

print(down_proj_weight_quant)

In [ ]:
print(model.model.norm.weight)
print(flh_fp16_model.model.norm.weight)
print(flh_model.model.norm.weight)